# ਪਾਠ 07 - ਯੋਜਨਾ ਬਣਾਉਣ ਦਾ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਹ ਨੋਟਬੁੱਕ ਮਾਈਕ੍ਰੋਸੋਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ AI ਏਜੰਟਾਂ ਲਈ **ਯੋਜਨਾ ਬਣਾਉਣ ਦਾ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ** ਦਰਸਾਉਂਦਾ ਹੈ।
ਤੁਸੀਂ ਸਿੱਖੋਗੇ ਕਿ ਕਿਵੇਂ ਜਟਿਲ ਯਾਤਰਾ ਦੀਆਂ ਵਿਨਤੀਆਂ ਨੂੰ ਰਚਨਾਤਮਕ ਉਪਕਾਰਜਾਂ ਵਿੱਚ ਵੰਡਣਾ ਹੈ, ਉਨ੍ਹਾਂ ਨੂੰ ਵਿਸ਼ੇਸ਼ਗਿਆ ਏਜੰਟਾਂ ਨੂੰ ਸੌਪਣਾ ਹੈ,
ਅਤੇ ਨਤੀਜੇ ਵਜੇਣ ਵਾਲੀ ਯੋਜਨਾ ਨੂੰ ਫਿਰ ਚਲਾਉਣਾ ਹੈ — ਸਾਰਾ ਕੁਝ ਪਾਇਡੈਂਟਿਕ ਮਾਡਲਾਂ ਤੋਂ ਚਲਾਇਆ ਗਿਆ ਰਚਨਾਤਮਕ ਆਉਟਪੁੱਟ ਵਰਤ ਕੇ।


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਕਾਰਜ ਵਿਭਾਜਨ

ਕਾਰਜ ਵਿਭਾਜਨ ਯੋਜਨਾ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਦਾ ਮੁੱਖ ਹੈ। ਇੱਕ ਹੀ ਏਜੰਟ ਨੂੰ
ਅਖੀਰ ਤੱਕ ਇੱਕ जਟਿਲ ਬੇਨਤੀ ਨੂੰ ਸੰਭਾਲਨ ਦੀ ਬਜਾਏ, ਅਸੀਂ ਸਮੱਸਿਆ ਨੂੰ ਛੋਟੇ, ਵਧੀਆ ਤਰੀਕੇ ਨਾਲ ਪਰਿਭਾਸ਼ਿਤ **ਉਪਕਾਰਜਾਂ** ਵਿੱਚ ਵੰਡਦੇ ਹਾਂ।
ਹਰ ਉਪਕਾਰਜ ਨੂੰ ਇੱਕ ਵਿਸ਼ੇਸ਼ਗਿਆ ਏਜੰਟ (ਜਿਵੇਂ ਕਿ, ਉਡਾਣਾਂ, ਹੋਟਲ, ਗਤਿਵਿਧੀਆਂ) ਨੂੰ ਸੌਂਪਿਆ ਜਾਂਦਾ ਹੈ ਜਿਸ ਦੀਆਂ ਸਪਸ਼ਟ
ਤਰਜੀਹਾਂ ਅਤੇ ਨਿਰਭਰਤਾ ਕ੍ਰਮ ਹੁੰਦੇ ਹਨ।

ਇਹ ਤਰੀਕਾ ਕਈ ਫਾਇਦੇ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ:
- **ਸਪਸ਼ਟਤਾ**: ਹਰ ਉਪਕਾਰਜ ਦੀ ਇੱਕ ਖਾਸ ਜ਼ਿੰਮੇਵਾਰੀ ਹੁੰਦੀ ਹੈ।
- **ਸਮਾਂਤਰਤਾ**: ਸਵਤੰਤਰ ਉਪਕਾਰਜ ਇੱਕਸਾਥ ਚਲ ਸਕਦੇ ਹਨ।
- **ਭਰੋਸੇਯੋਗਤਾ**: ਨਾਕਾਮੀਆਂ ਸਿਰਫ ਵਿਅਕਤੀਗਤ ਉਪਕਾਰਜਾਂ ਤੱਕ ਸੀਮਿਤ ਰਹਿੰਦੀਆਂ ਹਨ।
- **ਬਜਟ ਟ੍ਰੈਕਿੰਗ**: ਲਾਗਤਾਂ ਹਰ ਉਪਕਾਰਜ ਲਈ ਅੰਦਾਜ਼ੀ ਲਗਾਈਆਂ ਜਾਂਦੀਆਂ ਹਨ ਅਤੇ ਜੋੜੀਆਂ ਜਾਂਦੀਆਂ ਹਨ।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ਸਰੰਜਾਮਬੱਧ ਆਉਟਪੁੱਟ ਵਾਲਾ ਯੋਜਨਾ ਏਜੰਟ ਬਣਾਉਣਾ

ਯੋਜਨਾ ਏਜੰਟ ਇੱਕ **ਫਰੰਟ ਡੈਸਕ ਕੋਆਰਡੀਨੇਟਰ** ਵਜੋਂ ਕੰਮ ਕਰਦਾ ਹੈ। ਇੱਕ ਉੱਚ-ਸਤ੍ਹਰੀ ਯਾਤਰਾ ਬੇਨਤੀ ਮਿਲਣ 'ਤੇ ਇਹ
ਇੱਕ ਸਰੰਜਾਮਬੱਧ `TravelPlan` ਤਿਆਰ ਕਰਦਾ ਹੈ — ਬੇਨਤੀ ਨੂੰ ਉਪਕਾਰਜਾਂ ਵਿੱਚ ਵੰਡਦਾ ਹੈ, ਤਰਜੀحات ਸੈੱਟ ਕਰਦਾ ਹੈ,
ਅਤੇ ਸਰਲਤਾ ਨੂੰ ਪਛਾਣਦਾ ਹੈ ਤਾਂ ਜੋ ਇੱਕ ਕੋਈ ਸੰਭਾਲਨ ਵਾਲਾ ਜਾਂ ਕਾਰਜਨੂਮਾਈ ਪੱਧਰ ਕੰਮ ਨੂੰ ਜਾਰੀ ਰੱਖ ਸਕੇ।


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ਵਿਸ਼ੇਸ਼ਗਿਆਨ ਸੰਦਾਂ ਨਾਲ ਇੱਕ ਯੋਜਨਾ ਨੂੰ ਅਮਲ ਕਰਨਾ

ਜਦੋਂ ਫਰੰਟ ਡੈੱਸਕ ਏਜੰਟ ਇੱਕ ਸੰਗਠਿਤ ਯੋਜਨਾ ਤਿਆਰ ਕਰ ਲੈਂਦਾ ਹੈ, ਤਾਂ **ਕੌਨਸੀਅਰਜ ਏਜੰਟ** ਇਸਨੂੰ ਅਮਲ ਵਿੱਚ ਲਿਆਉਂਦਾ ਹੈ।
ਹਰ ਵਿਸ਼ੇਸ਼ਗਿਆਨ ਸੰਦ ਇੱਕ ਉਪਕੰਮ (ਫ਼ਲਾਈਟਾਂ, ਹੋਟਲ, ਗਤੀਵਿਧੀਆਂ) ਦੀ ਵਰਗਵਾਰ ਸੰਭਾਲ ਕਰਦਾ ਹੈ। ਕੌਨਸੀਅਰਜ
ਯੋਜਨਾ ਦੇ ਉਪਕੰਮਾਂ ਨੂੰ ਨਿਰਭਰਤਾ ਕ੍ਰਮ ਵਿੱਚ ਵਾਰੀ ਵਾਰੀ ਸੰਭਾਲਦਾ ਹੈ ਅਤੇ ਹਰ ਇੱਕ ਨੂੰ
ਸਹੀ ਸੰਦ ਨੂੰ ਭੇਜਦਾ ਹੈ।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ AI ਏਜੰਟਾਂ ਲਈ **ਯੋਜਨਾ ਨਿਰਦੇਸ਼ਨ ਪੈਟਰਨ** ਸਿੱਖਿਆ:

1. **ਕਾਰਜ ਵਿਭਾਜਨ** — ਇੱਕ ਫਰੰਟ ਡੈਸਕ ਯੋਜਨਾਬੱਧ ਏਜੰਟ ਇੱਕ ਜਟਿਲ ਯਾਤਰਾ ਬੇਨਤੀ ਨੂੰ Pydantic ਮਾਡਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ
   ਸੰਰਚਿਤ ਉਪਕਾਰਜਾਂ ਵਿੱਚ ਵੰਡੀਦਾ ਹੈ, ਪ੍ਰਤੀਕ ਤਰਤੇ ਪ੍ਰਾਥਮਿਕਤਾਵਾਂ ਅਤੇ ਆਧਾਰਾਂ ਨਾਲ ਹਰੇਕ ਨੂੰ ਕਿਸੇ ਵਿਸ਼ੇਸ਼ਜ ਾ ਏਜੰਟ ਨੂੰ ਸੌਂਪਦਾ ਹੈ।

2. **ਸੰਰਚਿਤ ਨਤੀਜਾ** — `response_format` ਦੇ ਗੁਜ਼ਾਰਨ ਦੇ ਜ਼ਰੀਏ ਏਜੰਟ ਇੱਕ ਵੈਧ `TravelPlan` ਵਸਤੂ ਵਾਪਸ ਕਰਦਾ ਹੈ
   ਬਜਾਏ ਖੁੱਲ੍ਹੇ ਰੂਪ ਦੇ ਟੈਕਸਟ ਦੇ, ਜਿਸ ਨਾਲ ਅਗਲੇ ਪ੍ਰਕਿਰਿਆਵਾਂ ਭਰੋਸੇਮੰਦ ਬਣ ਜਾਂਦੀਆਂ ਹਨ।
3. **ਯੋਜਨਾ ਕਾਰਜਾਂਵयन** — ਇੱਕ ਕਨਸੀਅਰਜ ਏਜੰਟ ਉਪਕਾਰਜਾਂ ਵਿੱਚ ਦੁਹਰਾਉਂਦਾ ਹੈ ਅਤੇ ਵਿਸ਼ੇਸ਼ਜ ਟੂਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰਦਾ ਹੈ
   (`book_flight`, `reserve_hotel`, `book_activity`) ਯੋਜਨਾ ਨੂੰ ਬਣਾਉਣ ਅਤੇ ਨਤੀਜੇ ਲਿਆਉਣ ਲਈ।

ਇਹ ਪੈਟਰਨ *ਕਿ ਕਰਨਾਂ ਹੈ* (ਯੋਜਨਾ ਬਣਾਉਣਾ) ਨੂੰ *ਕਿਵੇਂ ਕਰਨਾਂ ਹੈ* (ਕਾਰਜਾਂਵਯਨ) ਤੋਂ ਵੱਖਰਾ ਕਰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਏਜੰਟ
ਹੋਰ ਮਾਡੂਲਰ, ਲੱਛਣ ਯੋਗ, ਅਤੇ ਆਸਾਨੀ ਨਾਲ ਵਧਾਏ ਜਾ ਸਕਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
